# Session 5: Hypothesis Testing in the Clinical Laboratory

**Module 3: Programming for Biological Data**  
**Date:** January 19, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Understand the Normal Distribution and its role in reference intervals
2. Test data for normality using `shapiro.test()` and Q-Q plots
3. Perform one-sample, paired, and two-sample t-tests in R
4. Interpret p-values in the context of clinical decision-making

---

## Clinical Context

In medical laboratories, we constantly ask **comparative questions**:
- Is this new reagent lot equivalent to the old one?
- Is our analyzer calibrated correctly against the target value?
- Do viral loads differ between two patient groups?

**Hypothesis testing** provides a statistical framework to answer these questions objectively.

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL
# Run this cell first in every session
# ============================================

options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

ensure_loaded <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    message(paste("Installing", pkg, "..."))
    BiocManager::install(pkg, update = FALSE, ask = FALSE)
  }
  library(pkg, character.only = TRUE)
}

# Load packages for this session
ensure_loaded("ggplot2")

cat("✅ Setup complete! Ready for Session 5.")

---

# Part 1: The Normal Distribution (40 mins)

---

## 1.1 Why "Normal" Matters in the Lab

The **Normal (Gaussian) Distribution** is the foundation of:
- Reference intervals (Mean ± 2SD covers 95% of healthy population)
- Quality control charts (Levey-Jennings)
- Most statistical tests assume normality

### The Bell Curve Analogy

Think of measuring hemoglobin in 1000 healthy adults:
- Most values cluster around the center (mean)
- Fewer values at the extremes
- The shape is symmetric

In [ ]:
# Visualizing the Normal Distribution
# Simulating hemoglobin values in healthy adults (g/dL)

set.seed(123)  # For reproducibility
hemoglobin <- rnorm(n = 1000, mean = 14.0, sd = 1.5)

# Create a histogram
hist(hemoglobin, 
     breaks = 30,
     main = "Distribution of Hemoglobin in Healthy Adults",
     xlab = "Hemoglobin (g/dL)",
     col = "steelblue",
     border = "white")

# Add vertical lines for Mean and ±2SD
abline(v = mean(hemoglobin), col = "red", lwd = 2, lty = 1)
abline(v = mean(hemoglobin) + 2*sd(hemoglobin), col = "orange", lwd = 2, lty = 2)
abline(v = mean(hemoglobin) - 2*sd(hemoglobin), col = "orange", lwd = 2, lty = 2)

legend("topright", 
       legend = c("Mean", "Mean ± 2SD"),
       col = c("red", "orange"),
       lty = c(1, 2),
       lwd = 2)

### Key Properties

| Range | Coverage |
|-------|----------|
| Mean ± 1SD | ~68% of values |
| Mean ± 2SD | ~95% of values |
| Mean ± 3SD | ~99.7% of values |

**Clinical Application:** Reference intervals are typically Mean ± 2SD, capturing 95% of the healthy population.

In [ ]:
# Calculate reference interval (Mean ± 2SD)
mean_hb <- mean(hemoglobin)
sd_hb <- sd(hemoglobin)

lower_ref <- mean_hb - 2 * sd_hb
upper_ref <- mean_hb + 2 * sd_hb

cat("Hemoglobin Reference Interval:\n")
cat(sprintf("  Lower limit: %.1f g/dL\n", lower_ref))
cat(sprintf("  Upper limit: %.1f g/dL\n", upper_ref))

# What percentage falls within this range?
within_range <- sum(hemoglobin >= lower_ref & hemoglobin <= upper_ref) / length(hemoglobin) * 100
cat(sprintf("\n%.1f%% of values fall within reference interval\n", within_range))

---

## 1.2 Testing for Normality

Before using parametric tests (like t-tests), we should verify our data is approximately normal.

### Two Methods:
1. **Visual: Q-Q Plot** (Quantile-Quantile plot)
2. **Statistical: Shapiro-Wilk Test**

### Q-Q Plot: Visual Normality Check

A Q-Q plot compares your data against a theoretical normal distribution:
- If data is normal → points follow the diagonal line
- If data is skewed → points curve away from the line

In [ ]:
# Q-Q Plot for Normal Data
par(mfrow = c(1, 2))  # Side-by-side plots

# Normal data (hemoglobin)
qqnorm(hemoglobin, main = "Q-Q Plot: Normal Data")
qqline(hemoglobin, col = "red", lwd = 2)

# Skewed data (e.g., viral loads are often right-skewed)
set.seed(456)
viral_loads <- rexp(100, rate = 0.001)  # Exponential distribution
qqnorm(viral_loads, main = "Q-Q Plot: Skewed Data")
qqline(viral_loads, col = "red", lwd = 2)

par(mfrow = c(1, 1))  # Reset

### Shapiro-Wilk Test: Statistical Normality Check

The Shapiro-Wilk test formally tests normality:
- **Null hypothesis (H₀):** Data is normally distributed
- **If p < 0.05:** Reject H₀ → Data is NOT normal
- **If p ≥ 0.05:** Fail to reject H₀ → Data may be normal

In [ ]:
# Shapiro-Wilk Test

# Test hemoglobin data (should be normal)
cat("Testing Hemoglobin data:\n")
shapiro.test(hemoglobin[1:50])  # Use subset (Shapiro-Wilk works best with n < 5000)

cat("\n---\n\n")

# Test viral load data (should NOT be normal)
cat("Testing Viral Load data:\n")
shapiro.test(viral_loads)

### Clinical Relevance: CLSI C28-A3

The **CLSI C28-A3 guideline** for establishing reference intervals recommends:
- Testing data for normality before calculating reference intervals
- If data is not normal, use non-parametric methods (percentiles)
- For non-normal data, consider data transformation (e.g., log-transform viral loads)

In [ ]:
# Log-transformation of skewed data
# Viral loads are often log-transformed for analysis

log_viral <- log10(viral_loads + 1)  # +1 to avoid log(0)

par(mfrow = c(1, 2))

# Before transformation
hist(viral_loads, main = "Viral Load (raw)", xlab = "copies/mL", col = "coral")

# After transformation
hist(log_viral, main = "Viral Load (log10)", xlab = "log10(copies/mL)", col = "steelblue")

par(mfrow = c(1, 1))

# Test normality after transformation
cat("\nShapiro-Wilk test after log-transformation:\n")
shapiro.test(log_viral)

---

# Part 2: T-Tests — Comparing Means (40 mins)

---

## 2.1 The Logic of Hypothesis Testing

### The Courtroom Analogy

| Courtroom | Hypothesis Testing |
|-----------|--------------------|
| Defendant is innocent until proven guilty | H₀ (null hypothesis): No difference |
| Evidence must be convincing | p-value measures evidence against H₀ |
| "Beyond reasonable doubt" = conviction | p < 0.05 typically → Reject H₀ |
| "Not enough evidence" ≠ proven innocent | p ≥ 0.05 → Fail to reject (not "accept") H₀ |

### The P-value

**P-value = Probability of seeing results this extreme IF there's no real difference**

- p = 0.03 means: "There's only a 3% chance of seeing this difference by random chance"
- We typically use **α = 0.05** as our threshold (5% false positive rate)

---

## 2.2 One-Sample T-Test

**Question:** Does our sample mean differ from a known target value?

### Clinical Scenario: QC Material Verification

You receive a new lot of quality control material. The manufacturer states the target glucose value is **100 mg/dL**. You run 20 replicates. Does your analyzer measure at the target?

In [ ]:
# One-Sample T-Test Example
# QC Material Verification

set.seed(789)

# Your 20 QC measurements (slight positive bias)
qc_glucose <- c(101, 103, 99, 102, 100, 104, 98, 103, 101, 102,
                100, 105, 99, 101, 103, 102, 100, 104, 101, 103)

# Manufacturer's target value
target <- 100

cat("QC Glucose Measurements:\n")
cat("  Mean:", round(mean(qc_glucose), 2), "mg/dL\n")
cat("  SD:", round(sd(qc_glucose), 2), "mg/dL\n")
cat("  Target:", target, "mg/dL\n\n")

# Perform one-sample t-test
result <- t.test(qc_glucose, mu = target)
print(result)

### Interpreting the Output

```
t = test statistic (how many SDs from target)
df = degrees of freedom (n - 1)
p-value = probability of seeing this result if true mean = target
95% confidence interval = range of plausible true means
```

**Decision Rule:**
- If p < 0.05 → Significant bias from target
- If p ≥ 0.05 → No significant bias detected

In [ ]:
# Making a clinical decision

if (result$p.value < 0.05) {
  cat("⚠️ SIGNIFICANT BIAS DETECTED!\n")
  cat("The analyzer shows a statistically significant deviation from target.\n")
  cat(sprintf("Bias: %.2f mg/dL\n", mean(qc_glucose) - target))
} else {
  cat("✅ No significant bias detected.\n")
  cat("The analyzer is performing within acceptable limits.\n")
}

---

## 2.3 Paired T-Test

**Question:** Is there a difference between two measurements on the **same** samples?

### Clinical Scenario: Method Comparison Study

You're validating a new reagent. You measure 20 patient samples with both:
- **Method A:** Current in-use reagent
- **Method B:** New reagent lot

Are the results equivalent?

In [ ]:
# Paired T-Test Example
# Method Comparison: Current vs. New Reagent

# Glucose measurements (same 20 patients, both methods)
method_A <- c(95, 120, 88, 145, 102, 78, 110, 135, 92, 105,
              115, 98, 132, 87, 108, 125, 94, 118, 102, 138)

method_B <- c(97, 118, 90, 143, 105, 80, 108, 137, 94, 103,
              118, 100, 130, 89, 110, 123, 96, 120, 104, 136)

# Calculate differences
differences <- method_A - method_B

cat("Method Comparison Summary:\n")
cat("  Method A mean:", round(mean(method_A), 2), "mg/dL\n")
cat("  Method B mean:", round(mean(method_B), 2), "mg/dL\n")
cat("  Mean difference (A - B):", round(mean(differences), 2), "mg/dL\n")
cat("  SD of differences:", round(sd(differences), 2), "mg/dL\n")

In [ ]:
# Step 1: Check normality of differences (important for paired t-test)
cat("Checking normality of differences:\n")
shapiro.test(differences)

In [ ]:
# Step 2: Perform paired t-test
paired_result <- t.test(method_A, method_B, paired = TRUE)
print(paired_result)

In [ ]:
# Visualize the differences (Bland-Altman style)
mean_values <- (method_A + method_B) / 2

plot(mean_values, differences,
     main = "Difference Plot (Bland-Altman Style)",
     xlab = "Mean of A and B (mg/dL)",
     ylab = "Difference (A - B) (mg/dL)",
     pch = 19, col = "steelblue")

abline(h = mean(differences), col = "red", lwd = 2)  # Mean difference (bias)
abline(h = mean(differences) + 1.96 * sd(differences), col = "orange", lty = 2)  # Upper limit
abline(h = mean(differences) - 1.96 * sd(differences), col = "orange", lty = 2)  # Lower limit
abline(h = 0, col = "gray", lty = 3)  # Zero line

legend("topright",
       legend = c("Mean bias", "±1.96 SD limits"),
       col = c("red", "orange"),
       lty = c(1, 2), lwd = 2)

### Statistical vs. Clinical Significance

**Important concept for labs:**

Even if p < 0.05 (statistically significant), ask:
- **Is the difference clinically meaningful?**
- A 2 mg/dL difference in glucose might be statistically significant but clinically irrelevant
- Compare bias to **Total Allowable Error (TAE)** specifications

In [ ]:
# Clinical significance assessment
# Glucose TAE is typically ±10% or ±6 mg/dL (whichever is greater)

observed_bias <- abs(mean(differences))
clinical_threshold <- 6  # mg/dL

cat("\n=== Clinical Significance Assessment ===\n\n")
cat(sprintf("Observed bias: %.2f mg/dL\n", observed_bias))
cat(sprintf("Clinical threshold: %.1f mg/dL\n", clinical_threshold))
cat(sprintf("P-value: %.4f\n\n", paired_result$p.value))

if (paired_result$p.value < 0.05) {
  cat("Statistical: ⚠️ Significant difference detected\n")
} else {
  cat("Statistical: ✅ No significant difference\n")
}

if (observed_bias > clinical_threshold) {
  cat("Clinical: ⚠️ Bias exceeds clinical threshold — INVESTIGATE\n")
} else {
  cat("Clinical: ✅ Bias within acceptable clinical limits\n")
}

---

## 2.4 Two-Sample (Independent) T-Test

**Question:** Do two independent groups have different means?

### Clinical Scenario: Comparing Viral Loads

Do COVID-19 patients infected with **Delta** variant have different viral loads than those with **Omicron**?

In [ ]:
# Two-Sample T-Test Example
# Comparing viral loads between variants

set.seed(2024)

# Ct values (lower Ct = higher viral load)
# Delta tends to have lower Ct (higher viral load)
delta_ct <- c(18.5, 19.2, 17.8, 20.1, 18.9, 19.5, 17.2, 18.8, 19.8, 18.1,
              17.5, 19.0, 18.3, 20.5, 17.9, 18.6, 19.3, 17.8, 18.4, 19.1)

omicron_ct <- c(21.5, 22.8, 20.9, 23.1, 21.2, 22.5, 20.5, 21.8, 23.2, 22.0,
                21.0, 22.3, 21.7, 23.5, 20.8, 22.1, 21.9, 20.3, 22.6, 21.4)

cat("Ct Value Summary:\n")
cat(sprintf("  Delta: Mean = %.2f, SD = %.2f\n", mean(delta_ct), sd(delta_ct)))
cat(sprintf("  Omicron: Mean = %.2f, SD = %.2f\n", mean(omicron_ct), sd(omicron_ct)))

In [ ]:
# Visualize with boxplots
boxplot(delta_ct, omicron_ct,
        names = c("Delta", "Omicron"),
        main = "COVID-19 Ct Values by Variant",
        ylab = "Ct Value",
        col = c("coral", "steelblue"))

# Note: Lower Ct = Higher viral load

In [ ]:
# Perform two-sample t-test (independent samples)
two_sample_result <- t.test(delta_ct, omicron_ct)
print(two_sample_result)

In [ ]:
# Clinical interpretation
ct_difference <- mean(omicron_ct) - mean(delta_ct)

cat("\n=== Clinical Interpretation ===\n\n")
cat(sprintf("Mean Ct difference: %.2f cycles\n", ct_difference))
cat(sprintf("P-value: %.2e (scientific notation)\n\n", two_sample_result$p.value))

if (two_sample_result$p.value < 0.05) {
  cat("Conclusion: Ct values significantly differ between variants.\n")
  cat(sprintf("Omicron samples have %.1f higher Ct values on average.\n", ct_difference))
  cat("This suggests Delta has approximately 10x higher viral loads.\n")
  cat("(Each 3.3 Ct = 10-fold difference in viral copies)\n")
}

---

## Summary: Which T-Test to Use?

| Scenario | Test | R Code |
|----------|------|--------|
| Compare sample to known value | One-sample | `t.test(x, mu = value)` |
| Same subjects, two conditions | Paired | `t.test(x, y, paired = TRUE)` |
| Two independent groups | Two-sample | `t.test(x, y)` |

### Before ANY t-test:
1. Check normality (Shapiro-Wilk, Q-Q plot)
2. If not normal, consider transformation or non-parametric alternatives (Wilcoxon test)

---

# Key Takeaways

1. **Normal distribution** is foundational — always check before parametric tests
2. **Shapiro-Wilk test** (`shapiro.test()`) and **Q-Q plots** help assess normality
3. **P-value interpretation:** 
   - p < 0.05 → Statistically significant difference
   - p ≥ 0.05 → No significant difference detected (not "proven equal")
4. **Statistical ≠ Clinical significance** — always consider practical impact
5. T-tests compare **means** between groups or to a known value

---

## Now proceed to Tutorial 5! 🧪